In [196]:
import sys
import os
import warnings
from datetime import datetime, timezone, timedelta
from pathlib import Path
import pandas as pd
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
import pytz
import json
from zoneinfo import ZoneInfo
from pandas import Timestamp
from statsmodels.tsa.stattools import adfuller, kpss, acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.arima.model import ARIMA
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import r2_score
from scipy.stats import norm

current_dir = Path.cwd()
parent_dir = current_dir.parent
sys.path.insert(0, str(parent_dir))
from lib import *

MODEL_PATH=parent_dir / 'models' 
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 1000)

In [197]:
example_ticker = "KXBTC15M-26MAY040515-15"
lookback_minutes = 10000
series, event_dt = parse_kalshi_15m_event_ticker(example_ticker)
dt_only = get_ticker_datetime(example_ticker)
# crypto_at = datetime(2026,5,5,0,0,tzinfo=ZoneInfo('America/Chicago'))
crypto_at = datetime.now(tz=ZoneInfo('America/Chicago'))
df_api = get_market_data_from_api(series, crypto_at, lookback_minutes)
df_api = df_api.set_index('datetime')
df_api.head()

API Error Response: {'error': {'code': 'not_found', 'message': 'not found', 'service': 'query-exchange'}}
Error getting market candlesticks: 404 Client Error: Not Found for url: https://api.elections.kalshi.com/trade-api/v2/series/KXBTC15M/markets/KXBTC15M-26MAY210215-15/candlesticks?series_ticker=KXBTC15M&ticker=KXBTC15M-26MAY210215-15&start_ts=1779342900&end_ts=1779344100&period_interval=1&include_latest_before_start=True&limit=1000
API Error Response: {'error': {'code': 'not_found', 'message': 'not found', 'service': 'query-exchange'}}
Error getting market candlesticks: 404 Client Error: Not Found for url: https://api.elections.kalshi.com/trade-api/v2/series/KXBTC15M/markets/KXBTC15M-26MAY210230-30/candlesticks?series_ticker=KXBTC15M&ticker=KXBTC15M-26MAY210230-30&start_ts=1779343800&end_ts=1779345000&period_interval=1&include_latest_before_start=True&limit=1000
API Error Response: {'error': {'code': 'not_found', 'message': 'not found', 'service': 'query-exchange'}}
Error getting ma

,ticker,floor_strike,volume_fp,open_interest_fp,yes_ask_open_dollar,yes_ask_high_dollar,yes_ask_low_dollar,yes_ask_close_dollar,yes_bid_open_dollar,yes_bid_high_dollar,yes_bid_low_dollar,yes_bid_close_dollar,no_ask_open_dollar,no_ask_high_dollar,no_ask_low_dollar,no_ask_close_dollar,no_bid_open_dollar,no_bid_high_dollar,no_bid_low_dollar,no_bid_close_dollar
datetime,,,,,,,,,,,,,,,,,,,,
2026-05-14 21:31:00-05:00,KXBTC15M-26MAY142245-45,80890.83,54004.89,35892.38,0.999,0.999,0.42,0.47,0.00,0.54,0.00,0.45,0.55,0.46,1.00,1.00,0.53,0.001,0.58,0.001
2026-05-14 21:32:00-05:00,KXBTC15M-26MAY142245-45,80890.83,39241.03,66387.14,0.470,0.550,0.41,0.42,0.45,0.47,0.40,0.41,0.59,0.53,0.60,0.55,0.58,0.450,0.59,0.530
2026-05-14 21:33:00-05:00,KXBTC15M-26MAY142245-45,80890.83,52232.67,100668.22,0.420,0.460,0.22,0.27,0.41,0.44,0.21,0.26,0.74,0.56,0.79,0.59,0.73,0.540,0.78,0.580
2026-05-14 21:34:00-05:00,KXBTC15M-26MAY142245-45,80890.83,54155.93,141919.07,0.270,0.360,0.20,0.22,0.26,0.28,0.19,0.21,0.79,0.72,0.81,0.74,0.78,0.640,0.80,0.730
2026-05-14 21:35:00-05:00,KXBTC15M-26MAY142245-45,80890.83,45496.43,169124.64,0.220,0.260,0.17,0.24,0.21,0.24,0.15,0.23,0.77,0.76,0.85,0.79,0.76,0.740,0.83,0.780


In [198]:
df_crypto = get_crypto_past_minutes(series, crypto_at, lookback_minutes)
df_crypto['datetime'] = pd.to_datetime(df_crypto['datetime'])
df_crypto['datetime'] = df_crypto['datetime'].dt.tz_convert('America/Chicago')
df_crypto['datetime'] = df_crypto['datetime'].dt.floor('min')
df_crypto = df_crypto.set_index('datetime')
filter_timestamp = df_crypto[df_crypto.index.minute.isin([0,15,30,45])].index[0]
df_crypto = df_crypto[df_crypto.index >= filter_timestamp]
df_crypto.head()

,open,high,low,close,tick_count
datetime,,,,,
2026-05-09 13:45:00-05:00,80905.71,80911.16,80904.65,80906.61,53
2026-05-09 13:46:00-05:00,80898.19,80898.19,80898.19,80898.19,0
2026-05-09 13:47:00-05:00,80880.97,80880.97,80870.52,80870.52,10
2026-05-09 13:48:00-05:00,80876.48,80876.63,80876.48,80876.63,0
2026-05-09 13:49:00-05:00,80885.19,80885.19,80882.72,80883.92,14


In [199]:
df_merged = df_crypto.join(df_api, how='left')
df_merged = df_merged.dropna()
df_merged.head()

,open,high,low,close,tick_count,ticker,floor_strike,volume_fp,open_interest_fp,yes_ask_open_dollar,yes_ask_high_dollar,yes_ask_low_dollar,yes_ask_close_dollar,yes_bid_open_dollar,yes_bid_high_dollar,yes_bid_low_dollar,yes_bid_close_dollar,no_ask_open_dollar,no_ask_high_dollar,no_ask_low_dollar,no_ask_close_dollar,no_bid_open_dollar,no_bid_high_dollar,no_bid_low_dollar,no_bid_close_dollar
datetime,,,,,,,,,,,,,,,,,,,,,,,,,
2026-05-15 02:00:00-05:00,80568.66,80587.19,80565.18,80577.09,60,KXBTC15M-26MAY150300-00,80594.27,41833.32,91013.27,0.045,0.053,0.001,0.001,0.038,0.048,0.00,0.00,1.00,0.952,1.00,0.962,0.999,0.947,0.999,0.955
2026-05-15 02:01:00-05:00,80577.09,80600.04,80551.02,80589.77,60,KXBTC15M-26MAY150315-15,80577.58,15700.59,8867.51,0.989,0.989,0.450,0.570,0.000,0.580,0.00,0.56,0.44,0.420,1.00,1.000,0.430,0.011,0.550,0.011
2026-05-15 02:02:00-05:00,80589.77,80589.77,80589.10,80589.10,1,KXBTC15M-26MAY150315-15,80577.58,20482.47,23607.17,0.570,0.680,0.540,0.650,0.560,0.660,0.53,0.63,0.37,0.340,0.47,0.440,0.350,0.320,0.460,0.430
2026-05-15 02:03:00-05:00,80610.56,80614.39,80610.56,80614.39,9,KXBTC15M-26MAY150315-15,80577.58,21983.86,38680.62,0.650,0.700,0.640,0.690,0.630,0.680,0.62,0.68,0.32,0.320,0.38,0.370,0.310,0.300,0.360,0.350
2026-05-15 02:04:00-05:00,80608.40,80608.40,80608.40,80608.40,0,KXBTC15M-26MAY150315-15,80577.58,22549.72,51566.55,0.690,0.690,0.570,0.570,0.680,0.680,0.56,0.56,0.44,0.320,0.44,0.320,0.430,0.310,0.430,0.310


In [200]:
df_calc = df_merged

In [201]:
def rsi_calc(series, periods=14):
    delta = series.diff()
    gain = (delta.where(delta>0,0)).rolling(periods).mean()
    loss = (delta.where(delta<0,0)).rolling(periods).mean()
    rs = gain / loss
    rsi = 100 - (100/(1+rs))
    return rsi

In [202]:
def macd_calc(series, fast=14, slow=26, signal=13):
    ema_fast = series.ewm(span=fast, adjust=False).mean()
    ema_slow = series.ewm(span=slow, adjust=False).mean()
    macd_line = ema_fast - ema_slow
    signal_line = macd_line.ewm(span=signal, adjust=False).mean()
    histogram = macd_line - signal_line
    return macd_line, signal_line, histogram

In [203]:
research_side = 'yes'
df_calc['rsi'] = rsi_calc(df_calc['close'], periods=5) 
df_calc['macd'], df_calc['signal_line'], _ = macd_calc(df_calc['close'], 4, 9, 2) 
df_calc[research_side + '_dist'] = df_calc['close'] - df_calc['floor_strike']
df_calc['log_return'] = np.log(df_calc['close'] / df_calc['close'].shift(1))
df_calc['m3_log_return'] = df_calc['log_return'].rolling(3).std()
df_calc['m5_log_return'] = df_calc['log_return'].rolling(5).std()
df_calc['ma3'] = df_calc['close'].rolling(3).mean()
df_calc['ma5'] = df_calc['close'].rolling(5).mean()
df_calc['ma3_vs_strike'] = (df_calc['ma3'] - df_calc['floor_strike'])/df_calc['floor_strike'] * 100
df_calc['ma5_vs_strike'] = (df_calc['ma5'] - df_calc['floor_strike'])/df_calc['floor_strike'] * 100
df_calc[research_side + '_dist_pct'] = df_calc[research_side + '_dist'] / df_calc['floor_strike'] * 100
df_calc['m1_' + research_side + '_dist_momentum'] = df_calc[research_side + '_dist'] - df_calc[research_side + '_dist'].shift(1)
df_calc['m3_' + research_side + '_dist_momentum'] = df_calc[research_side + '_dist'] - df_calc[research_side + '_dist'].shift(3)
df_calc['m5_' + research_side + '_dist_momentum'] = df_calc[research_side + '_dist'] - df_calc[research_side + '_dist'].shift(5)
df_calc['time_decay'] = np.where(df_calc.index.minute % 15 == 0, 0, 15 - df_calc.index.minute % 15)
df_calc['hour'] = df_calc.index.hour
df_calc['minute'] = df_calc.index.minute
df_calc[research_side + '_spread'] = df_calc[research_side + '_ask_close_dollar'] - df_calc[research_side + '_bid_close_dollar']
df_calc['volume_surge'] = df_calc['volume_fp'] / df_calc['volume_fp'].rolling(5).mean()
df_calc['ask_delta'] = df_calc[research_side + '_ask_low_dollar'].diff()
df_calc['bid_delta'] = df_calc[research_side + '_bid_high_dollar'].diff()
df_calc['ask_std'] = df_calc[research_side + '_ask_low_dollar'].rolling(3).std()
df_calc['bid_std'] = df_calc[research_side + '_bid_high_dollar'].rolling(3).std()
# df_calc['oi_change'] = df_calc['open_interest_fp'] - df_calc['open_interest_fp'].shift(1)

In [204]:
df_calc = df_calc.dropna()

In [205]:
df_calc.head()

,open,high,low,close,tick_count,ticker,floor_strike,volume_fp,open_interest_fp,yes_ask_open_dollar,yes_ask_high_dollar,yes_ask_low_dollar,yes_ask_close_dollar,yes_bid_open_dollar,yes_bid_high_dollar,yes_bid_low_dollar,yes_bid_close_dollar,no_ask_open_dollar,no_ask_high_dollar,no_ask_low_dollar,no_ask_close_dollar,no_bid_open_dollar,no_bid_high_dollar,no_bid_low_dollar,no_bid_close_dollar,rsi,macd,signal_line,yes_dist,log_return,m3_log_return,m5_log_return,ma3,ma5,ma3_vs_strike,ma5_vs_strike,yes_dist_pct,m1_yes_dist_momentum,m3_yes_dist_momentum,m5_yes_dist_momentum,time_decay,hour,minute,yes_spread,volume_surge,ask_delta,bid_delta,ask_std,bid_std
datetime,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2026-05-15 02:05:00-05:00,80597.05,80616.00,80593.11,80601.44,48,KXBTC15M-26MAY150315-15,80577.58,14710.76,59021.35,0.57,0.68,0.56,0.66,0.56,0.66,0.55,0.65,0.35,0.34,0.45,0.44,0.34,0.32,0.44,0.43,-2.787812,7.398026,7.728321,23.86,-0.000086,0.000228,0.000172,80608.076667,80600.620,0.037848,0.028594,0.029611,-6.96,12.34,41.04,10,2,5,0.01,0.770783,-0.01,-0.02,0.043589,0.011547
2026-05-15 02:06:00-05:00,80608.72,80610.59,80604.85,80609.77,10,KXBTC15M-26MAY150315-15,80577.58,24001.83,71386.77,0.66,0.80,0.62,0.75,0.65,0.79,0.61,0.73,0.27,0.21,0.39,0.35,0.25,0.20,0.38,0.34,-2.468429,7.590301,7.636308,32.19,0.000103,0.000106,0.000166,80606.536667,80604.620,0.035936,0.033558,0.039949,8.33,-4.62,20.00,9,2,6,0.02,1.156953,0.06,0.13,0.032146,0.070000
2026-05-15 02:07:00-05:00,80632.33,80641.06,80629.51,80637.50,15,KXBTC15M-26MAY150315-15,80577.58,19014.17,80120.98,0.75,0.85,0.70,0.80,0.73,0.84,0.69,0.79,0.21,0.16,0.31,0.27,0.20,0.15,0.30,0.25,-4.737452,12.621370,10.959682,59.92,0.000344,0.000216,0.000205,80616.236667,80614.300,0.047974,0.045571,0.074363,27.73,29.10,48.40,8,2,7,0.01,0.929694,0.08,0.05,0.070238,0.092916
2026-05-15 02:08:00-05:00,80653.55,80653.55,80653.55,80653.55,0,KXBTC15M-26MAY150315-15,80577.58,17218.25,85494.55,0.80,0.82,0.76,0.81,0.79,0.81,0.75,0.80,0.20,0.19,0.25,0.21,0.19,0.18,0.24,0.20,-4.023938,17.236573,15.144276,75.97,0.000199,0.000121,0.000183,80633.606667,80622.132,0.069531,0.055291,0.094282,16.05,52.11,39.16,7,2,8,0.01,0.883035,0.06,-0.03,0.070238,0.025166
2026-05-15 02:09:00-05:00,80638.33,80638.33,80638.33,80638.33,0,KXBTC15M-26MAY150315-15,80577.58,27167.53,87316.34,0.81,0.82,0.34,0.40,0.80,0.80,0.33,0.37,0.63,0.20,0.67,0.20,0.60,0.18,0.66,0.19,-2.349414,15.028945,15.067388,60.75,-0.000189,0.000275,0.000215,80643.126667,80628.118,0.081346,0.062720,0.075393,-15.22,28.56,29.93,6,2,9,0.03,1.330274,-0.42,-0.01,0.227156,0.020817


In [206]:
def agg_data_function(df, column, *data_cents):
    results = {cent: [] for cent in data_cents}
    triggered = set() 
    
    for i in range(len(df)):
        row = df.iloc[i]
        
        if row['time_decay'] == 0:
            continue
        
        current_ticker = row['ticker']
        
        for cent in data_cents:
            if (current_ticker, cent) in triggered:
                continue
                
            if float(row[column + '_ask_low_dollar']) < float(cent):
                # print(f"trigger: {column + '_ask_low_dollar'}: {row[column + '_ask_low_dollar']}")
                triggered.add((current_ticker, cent)) 
                
                high_price = 0
                low_price = 1
                
                for j in range(1, 16):
                    if i + j >= len(df):
                        break
                    next_row = df.iloc[i + j]
                    high_price = max(next_row[column + '_bid_high_dollar'], high_price)
                    low_price = min(next_row[column + '_ask_low_dollar'], low_price)
                    if next_row['ticker'] != current_ticker or next_row['time_decay'] == 0:
                        break
                
                tmp_dict = row.to_dict()
                tmp_dict['subsequent_high'] = high_price
                tmp_dict['subsequent_low'] = low_price
                tmp_dict['reached_30'] = 1 if high_price >= 0.30 else 0
                tmp_dict['reached_40'] = 1 if high_price >= 0.40 else 0
                tmp_dict['reached_50'] = 1 if high_price >= 0.50 else 0
                tmp_dict['reached_60'] = 1 if high_price >= 0.60 else 0
                tmp_dict['reached_70'] = 1 if high_price >= 0.7 else 0
                tmp_dict['reached_90'] = 1 if high_price >= 0.9 else 0
                results[cent].append(tmp_dict)
                
    
    return results

In [207]:
res=agg_data_function(df_calc, research_side, *[0.05,0.1,0.15,0.2,0.25,0.3,0.35])

In [208]:
# win rate analysis
comb_05 = pd.DataFrame(res[0.05])
comb_10 = pd.DataFrame(res[0.10])
comb_15 = pd.DataFrame(res[0.15])
comb_20 = pd.DataFrame(res[0.2])
comb_25 = pd.DataFrame(res[0.25])
comb_30 = pd.DataFrame(res[0.3])
comb_35 = pd.DataFrame(res[0.35])

In [209]:
def get_protential_pnl(df, entry_price, *exit_price):
    for price in exit_price:
        col_name = f"reached_{int(price * 100)}"
        if col_name in df.columns:
            rate = (df[col_name] == 1).sum() / len(df)
            pnl = rate * (float(price) - float(entry_price))
            print(f"Entry: {entry_price}, Exit: {price}, win rate: {rate:.2%}, PNL: {pnl:.4f}")

In [210]:
def evaluate_feature(good_mean, good_std, bad_mean, bad_std):
    diff = good_mean - bad_mean
    avg_std = (good_std + bad_std) / 2
    
    if avg_std < 1e-10:
        return None
    
    ratio = abs(diff) / avg_std
    
    if ratio > 0.5:
        grade = 'strong'
    elif ratio > 0.3:
        grade = 'medium'
    elif ratio > 0.15:
        grade = 'weak'
    else:
        grade = 'useless'
    
    return ratio, grade

In [248]:
good_hours = [10,11,12]

get_protential_pnl(comb_05[comb_05['hour'].isin(good_hours)],0.05,*[0.4,0.5,0.6,0.7,0.9])
get_protential_pnl(comb_10[comb_10['hour'].isin(good_hours)],0.10,*[0.4,0.5,0.6,0.7,0.9])
get_protential_pnl(comb_15[comb_15['hour'].isin(good_hours)],0.15,*[0.4,0.5,0.6,0.7,0.9])
get_protential_pnl(comb_20[comb_20['hour'].isin(good_hours)],0.20,*[0.4,0.5,0.6,0.7,0.9])
get_protential_pnl(comb_25[comb_25['hour'].isin(good_hours)],0.25,*[0.4,0.5,0.6,0.7,0.9])
get_protential_pnl(comb_30[comb_30['hour'].isin(good_hours)],0.30,*[0.4,0.5,0.6,0.7,0.9])
get_protential_pnl(comb_35[comb_35['hour'].isin(good_hours)],0.35,*[0.4,0.5,0.6,0.7,0.9])

Entry: 0.05, Exit: 0.4, win rate: 6.25%, PNL: 0.0219
Entry: 0.05, Exit: 0.5, win rate: 6.25%, PNL: 0.0281
Entry: 0.05, Exit: 0.6, win rate: 0.00%, PNL: 0.0000
Entry: 0.05, Exit: 0.7, win rate: 0.00%, PNL: 0.0000
Entry: 0.05, Exit: 0.9, win rate: 0.00%, PNL: 0.0000
Entry: 0.1, Exit: 0.4, win rate: 31.82%, PNL: 0.0955
Entry: 0.1, Exit: 0.5, win rate: 27.27%, PNL: 0.1091
Entry: 0.1, Exit: 0.6, win rate: 22.73%, PNL: 0.1136
Entry: 0.1, Exit: 0.7, win rate: 22.73%, PNL: 0.1364
Entry: 0.1, Exit: 0.9, win rate: 18.18%, PNL: 0.1455
Entry: 0.15, Exit: 0.4, win rate: 34.78%, PNL: 0.0870
Entry: 0.15, Exit: 0.5, win rate: 30.43%, PNL: 0.1065
Entry: 0.15, Exit: 0.6, win rate: 26.09%, PNL: 0.1174
Entry: 0.15, Exit: 0.7, win rate: 26.09%, PNL: 0.1435
Entry: 0.15, Exit: 0.9, win rate: 21.74%, PNL: 0.1630
Entry: 0.2, Exit: 0.4, win rate: 48.15%, PNL: 0.0963
Entry: 0.2, Exit: 0.5, win rate: 40.74%, PNL: 0.1222
Entry: 0.2, Exit: 0.6, win rate: 37.04%, PNL: 0.1481
Entry: 0.2, Exit: 0.7, win rate: 33.33%, 

In [249]:
feature_cols = [  
    'ask_delta',
    'bid_delta',
    'ask_std',
    'bid_std',
    research_side + '_dist',         
    'log_return',  
    'm3_log_return',
    'm5_log_return',  
    'ma3_vs_strike',
    'ma5_vs_strike',            
    research_side + '_dist_pct',
    'm1_' + research_side + '_dist_momentum',
    'm3_' + research_side + '_dist_momentum',
    'm5_' + research_side + '_dist_momentum',
    'time_decay',
    'hour',
    'minute',
    research_side + '_spread',
    'volume_surge',
    'rsi',
    'macd',
    'signal_line',
]

comb_list = [0.05, 0.1, 0.15, 0.2, 0.25,0.3,0.35]
reached_list = ['reached_30','reached_40', 'reached_50', 'reached_60', 'reached_70', 'reached_90']

for comb in comb_list: 
    for reached in reached_list:
        df_results = pd.DataFrame(res[comb])
        df_results = df_results[df_results['hour'].isin(good_hours)]
        df_expected = df_results[reached]
        
        # Separate good and bad outcomes
        good = df_results[df_expected == 1]
        bad = df_results[df_expected == 0]
        
        # Prior probabilities
        prior_good = len(good) / len(df_results)
        prior_bad = len(bad) / len(df_results)
        print(f"Comb is {comb}, reached is {reached}")
        print(f"Prior P(Good) = {prior_good:.4f}, P(Bad) = {prior_bad:.4f}")
        likelihoods = {}
        good_ratio_count = 0
        for col in feature_cols:
            good_mean, good_std = good[col].mean(), good[col].std()
            bad_mean, bad_std = bad[col].mean(), bad[col].std()
            
            likelihoods[col] = {
                'good': (good_mean, good_std),
                'bad': (bad_mean, bad_std),
            }
            
            diff = good_mean - bad_mean
            ratio, grade = evaluate_feature(good_mean, good_std, bad_mean, bad_std)
            if grade == 'strong' or grade == 'medium':
                good_ratio_count += 1
            print(f"{grade} -> {col}: Good={good_mean:.4f}, Bad={bad_mean:.4f}, ratio={ratio:.4f}")
        print(f"Total Good Ratio is {good_ratio_count}\n")

Comb is 0.05, reached is reached_30
Prior P(Good) = 0.0625, P(Bad) = 0.9375
useless -> ask_delta: Good=-0.1180, Bad=-0.1391, ratio=nan
useless -> bid_delta: Good=-0.0200, Bad=-0.1181, ratio=nan
useless -> ask_std: Good=0.0681, Bad=0.0968, ratio=nan
useless -> bid_std: Good=0.0700, Bad=0.0937, ratio=nan
useless -> yes_dist: Good=-85.4000, Bad=-74.8240, ratio=nan
useless -> log_return: Good=-0.0001, Bad=-0.0003, ratio=nan
useless -> m3_log_return: Good=0.0000, Bad=0.0005, ratio=nan
useless -> m5_log_return: Good=0.0005, Bad=0.0006, ratio=nan
useless -> ma3_vs_strike: Good=-0.1015, Bad=-0.0849, ratio=nan
useless -> ma5_vs_strike: Good=-0.0830, Bad=-0.0720, ratio=nan
useless -> yes_dist_pct: Good=-0.1108, Bad=-0.0973, ratio=nan
useless -> m1_yes_dist_momentum: Good=-8.3500, Bad=-20.7747, ratio=nan
useless -> m3_yes_dist_momentum: Good=-23.9100, Bad=-22.8947, ratio=nan
useless -> m5_yes_dist_momentum: Good=-163.1500, Bad=-73.9507, ratio=nan
useless -> time_decay: Good=4.0000, Bad=3.4667, ra

In [266]:
df_name = 0.10
df_result_name = 'reached_90'

df_results = pd.DataFrame(res[df_name])
df_results = df_results[df_results['hour'].isin(good_hours)]
df_expected = df_results[df_result_name]

# Separate good and bad outcomes
good = df_results[df_expected == 1]
bad = df_results[df_expected == 0]

# Prior probabilities
prior_good = len(good) / len(df_results)
prior_bad = len(bad) / len(df_results)
likelihoods = {}
params = {}

for col in feature_cols:
    good_mean, good_std = good[col].mean(), good[col].std()
    bad_mean, bad_std = bad[col].mean(), bad[col].std()
    
    likelihoods[col] = {
        'good': (good_mean, good_std),
        'bad': (bad_mean, bad_std),
    }
    
    # diff = good_mean - bad_mean
    ratio, grade = evaluate_feature(good_mean, good_std, bad_mean, bad_std)
    if grade == 'strong' or grade == 'medium':
        print(f"{grade} -> {col}: Good={good_mean:.4f}, Bad={bad_mean:.4f}, Ratio={ratio:.4f}")
        params[col] = {'good': (good_mean, good_std), 'bad': (bad_mean, bad_std),}
# for col in feature_cols:
#     g_mean, g_std = likelihoods[col]['good']
#     b_mean, b_std = likelihoods[col]['bad']
#     params[col] = {
#         'good': (g_mean, g_std),
#         'bad': (b_mean, b_std)
#     }
    
params['period'] = {
    'good': (prior_good,),
    'bad': (prior_bad,)
}
print(f"good is: {prior_good}, bad is: {prior_bad}")

strong -> bid_std: Good=0.0555, Bad=0.1420, Ratio=1.2382
strong -> log_return: Good=0.0000, Bad=-0.0004, Ratio=0.9728
strong -> m3_log_return: Good=0.0003, Bad=0.0005, Ratio=0.8029
strong -> m5_log_return: Good=0.0003, Bad=0.0005, Ratio=1.1722
strong -> m1_yes_dist_momentum: Good=1.1200, Bad=-28.0622, Ratio=0.9760
medium -> m5_yes_dist_momentum: Good=-64.1725, Bad=-86.5944, Ratio=0.3238
medium -> time_decay: Good=5.7500, Bad=4.6667, Ratio=0.3624
medium -> hour: Good=11.2500, Bad=10.8889, Ratio=0.4035
strong -> yes_spread: Good=0.0103, Bad=0.0063, Ratio=0.5824
good is: 0.18181818181818182, bad is: 0.8181818181818182


In [267]:
# type 1 and 2 errors

type1 = 0 
type2 = 0  

total_good = 0
total_bad = 0

feature_names = [name for name in params.keys() if name != 'period']
def predict(*args):
    # Start from prior odds
    log_odds = np.log(params['period']['good'][0] / params['period']['bad'][0])
    
    for name, x in zip(feature_names, args):
        g_m, g_s = params[name]['good']
        b_m, b_s = params[name]['bad']
        
        # Likelihood ratio
        p_g = norm.pdf(x, g_m, g_s + 1e-10)
        p_b = norm.pdf(x, b_m, b_s + 1e-10)
        
        if p_b > 0:
            log_odds += np.log(p_g / p_b)
    
    prob = 1 / (1 + np.exp(-log_odds))
    return prob

for threshold in [0.1, 0.15, 0.2, 0.25, 0.30, 0.35, 0.40, 0.45]:
    type1, type2 = 0, 0
    total_good, total_bad = 0, 0
    
    for index, row in df_results.iterrows(): 
        p = predict(*(row[x] for x in params if x != 'period')) 
        
        actual = row[df_result_name]
        
        if actual == 1:
            total_good += 1
            if p < threshold:
                type1 += 1
        if actual == 0:
            total_bad += 1
            if p >= threshold:
                type2 += 1
    
    bought_good = total_good - type1
    bought_bad = type2
    
    type1_rate = type1 / total_good
    type2_rate = type2 / total_bad
    
    # Precision & Recall
    precision = bought_good / (bought_good + bought_bad) if (bought_good + bought_bad) > 0 else 0
    recall = 1 - type1_rate
    
    # F1
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    
    print(f"Threshold={threshold:.2f}: Type1={type1_rate:.2f}, Type2={type2_rate:.2f}, F1={f1:.4f}, Precision={precision:.4f}, Recall={recall:.4f}")

Threshold=0.10: Type1=0.00, Type2=0.22, F1=0.6667, Precision=0.5000, Recall=1.0000
Threshold=0.15: Type1=0.00, Type2=0.17, F1=0.7273, Precision=0.5714, Recall=1.0000
Threshold=0.20: Type1=0.00, Type2=0.17, F1=0.7273, Precision=0.5714, Recall=1.0000
Threshold=0.25: Type1=0.25, Type2=0.17, F1=0.6000, Precision=0.5000, Recall=0.7500
Threshold=0.30: Type1=0.25, Type2=0.17, F1=0.6000, Precision=0.5000, Recall=0.7500
Threshold=0.35: Type1=0.25, Type2=0.17, F1=0.6000, Precision=0.5000, Recall=0.7500
Threshold=0.40: Type1=0.25, Type2=0.17, F1=0.6000, Precision=0.5000, Recall=0.7500
Threshold=0.45: Type1=0.25, Type2=0.17, F1=0.6000, Precision=0.5000, Recall=0.7500


In [269]:
# export model

def write_model_to_json(parameters: dict, filepath: str = None):
    if filepath is None:
        filepath = MODEL_PATH / 'yes_bayesian.json'
    
    params_serializable = {}
    for feature, values in parameters.items():
        params_serializable[feature] = {
            'good': list(values['good']),
            'bad': list(values['bad'])
        }
    
    with open(filepath, 'w') as f:
        json.dump(params_serializable, f, indent=2)
    
    print(f"Model saved to {filepath}")


def load_model_from_json(filepath: str = None):
    if filepath is None:
        filepath = MODEL_PATH / 'yes_bayesian.json'
    
    with open(filepath, 'r') as f:
        params_serializable = json.load(f)
    
    # Convert lists back to tuples
    parameters = {}
    for feature, values in params_serializable.items():
        parameters[feature] = {
            'good': tuple(values['good']),
            'bad': tuple(values['bad'])
        }
    
    return parameters


# Correct dict syntax
# params = {
#     'period': {
#         'good': (0.4138,),
#         'bad': (0.5862,)
#     },
#     'm1_yes_dist_momentum': {
#         'good': (-17.5070, 51.8947),
#         'bad':  (-19.7738, 48.4898)
#     },
#     'm3_yes_dist_momentum': {
#         'good': (-29.0380, 82.5982),
#         'bad':  (-32.8337, 73.5574)
#     },
#     'm5_yes_dist_momentum': {
#         'good': (-31.1500, 93.6206),
#         'bad':  (-37.5775, 85.6289)
#     },
#     'yes_dist': {
#         'good': (-19.9004, 29.5222),
#         'bad':  (-22.8592, 38.3313)
#     },
# }

# Write
write_model_to_json(params)
print(params)

Model saved to /Users/yingxie/Documents/Git/Quant/Kalshi/btc_15_strategy/models/yes_bayesian.json
{'bid_std': {'good': (0.05551869136485644, 0.03959390933742282), 'bad': (0.14202164912011886, 0.10013451382278316)}, 'log_return': {'good': (1.3120066927865888e-05, 0.00033306754451020574), 'bad': (-0.00036494169821754793, 0.0004441631194665841)}, 'm3_log_return': {'good': (0.0002830137026006047, 0.00015543394372929706), 'bad': (0.0005115971174848278, 0.00041393473308045807)}, 'm5_log_return': {'good': (0.00027138359618607895, 0.00012090859339240252), 'bad': (0.0005079018720299936, 0.000282650453974953)}, 'm1_yes_dist_momentum': {'good': (1.1200000000026193, 25.691137252631815), 'bad': (-28.062222222222772, 34.110125382278305)}, 'm5_yes_dist_momentum': {'good': (-64.17249999999694, 65.80278685841313), 'bad': (-86.59444444444509, 72.70699773929562)}, 'time_decay': {'good': (5.75, 2.8722813232690143), 'bad': (4.666666666666667, 3.1059714782221377)}, 'hour': {'good': (11.25, 0.957427107756338

In [253]:
# hour research
reached_list = [50,60,70,90]
time_dict = {str(x): 0 for x in range(24)}
test_df = comb_15
test_entry_price = 15

for hour in range(24):
    for reached in reached_list:
        df_h = test_df[test_df['hour'] == hour]
        # if len(df_h) < 2:
        #     continue
        rate = df_h['reached_' + str(reached)].mean()
        ev = rate * ((reached - test_entry_price) / 100)  - (1-rate) * test_entry_price / 100
        time_dict[str(hour)] += ev

sorted_items = sorted(time_dict.items(), key=lambda x: int(x[0]), reverse=True)
for key, item in sorted_items:
    print(f"Hour: {key}, EV: {item:.3f}")

Hour: 23, EV: nan
Hour: 22, EV: nan
Hour: 21, EV: nan
Hour: 20, EV: nan
Hour: 19, EV: -0.600
Hour: 18, EV: -0.600
Hour: 17, EV: -0.420
Hour: 16, EV: 0.092
Hour: 15, EV: 0.480
Hour: 14, EV: -0.057
Hour: 13, EV: -0.462
Hour: 12, EV: 0.412
Hour: 11, EV: 0.114
Hour: 10, EV: -0.263
Hour: 9, EV: -0.185
Hour: 8, EV: -0.313
Hour: 7, EV: -0.555
Hour: 6, EV: -0.136
Hour: 5, EV: -0.244
Hour: 4, EV: -0.195
Hour: 3, EV: -0.441
Hour: 2, EV: 0.000
Hour: 1, EV: -0.100
Hour: 0, EV: nan


In [225]:
# start research
trade_hour = [x for x in range(23)]
reached_dict = {str(x): 0 for x in reached_list}
for hour in trade_hour:
    for reached in reached_list:
        df_h = test_df[test_df['hour'] == hour]
        if len(df_h) < 5:
            continue
        rate = df_h['reached_' + str(reached)].mean()
        ev = rate * ((reached - test_entry_price)/100 - 0.1)  - (1-rate) * test_entry_price / 100
        reached_dict[str(reached)] += ev

for key, item in reached_dict.items():
    print(f"Reached: {key}, EV: {item:.3f}")

Reached: 50, EV: -0.948
Reached: 60, EV: -0.843
Reached: 70, EV: -0.814
Reached: 90, EV: -0.716


In [230]:
# final hour research
trade_hour = [x for x in range(23)]
for hour in trade_hour:
    df_h = test_df[test_df['hour'] == hour]
    if len(df_h) < 5:
        continue
    rate = df_h['reached_50'].mean()
    ev = rate * (0.5 - 0.1)  - (1-rate) * 0.10
    print(f"{hour:02d}:00 {len(df_h):3d} trades win {rate:.0%} | EV {ev:.2f}")

02:00  12 trades win 25% | EV 0.02
03:00  17 trades win 6% | EV -0.07
04:00  19 trades win 32% | EV 0.06
05:00  18 trades win 22% | EV 0.01
06:00  14 trades win 21% | EV 0.01
07:00  11 trades win 9% | EV -0.05
08:00  15 trades win 20% | EV 0.00
09:00  13 trades win 15% | EV -0.02
10:00   8 trades win 12% | EV -0.04
11:00   7 trades win 43% | EV 0.11
12:00   8 trades win 38% | EV 0.09
13:00  13 trades win 8% | EV -0.06
14:00   7 trades win 29% | EV 0.04
15:00  10 trades win 40% | EV 0.10
16:00  12 trades win 33% | EV 0.07
17:00  10 trades win 10% | EV -0.05
